In [1]:
# =============================================================================
# LLM Call Template
# Model: TinyLlama (可替換成其他 HuggingFace 模型)
# =============================================================================


# -----------------------------------------------------------------------------
# Cell 1：安裝套件（只需執行一次）
# -----------------------------------------------------------------------------

import os
os.system("pip install -q transformers accelerate sentencepiece")

0

In [2]:
# -----------------------------------------------------------------------------
# Cell 2：載入模型（只需執行一次，之後重複測試不用重跑）
# -----------------------------------------------------------------------------

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ⚙️ 設定區（修改這裡來切換模型或行為）
MODEL_NAME     = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 替換成任何 HuggingFace Chat 模型
MAX_NEW_TOKENS = 200
DO_SAMPLE      = False  # True = 隨機生成（創意）；False = 確定性生成（精準）

def load_model(model_name: str = MODEL_NAME):
    """載入 tokenizer 與模型，回傳 (tokenizer, model)"""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

def llm(messages: list) -> str:
    """
    傳入完整對話歷史，回傳模型生成的文字。

    Args:
        messages: [{"role": "system/user/assistant", "content": "..."}]

    Returns:
        模型生成的回覆字串
    """
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

tokenizer, model = load_model()
print("✅ 模型載入完成")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ 模型載入完成


In [3]:
# -----------------------------------------------------------------------------
# Cell 3：執行（修改 SYSTEM_PROMPT / USER_PROMPT 後重複跑這格）
# -----------------------------------------------------------------------------

# ✏️ 修改這裡
SYSTEM_PROMPT = "You are a helpful assistant. Answer concisely."
USER_PROMPT   = "15 * 12 是多少？"

MESSAGES = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": USER_PROMPT}
]

print(f"問題：{USER_PROMPT}")
print(f"回答：{llm(MESSAGES)}")


問題：15 * 12 是多少？
回答：Yes, 15 * 12 is 180.
